> 在线讲义：[第 9 章 · 最优价值与 Bellman 最优方程：把"平均"换成"挑最好的"](https://wikieden.github.io/rl-from-scratch/lessons/09-optimal.html) · 本书《从零学强化学习》

# 第 9 章 · 最优价值与 Bellman 最优方程：把"平均"换成"挑最好的"

本章回答：什么叫"最好的策略"？最好的策略下价值是多少？知道最优价值以后，怎么直接读出该做什么？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/09-optimal.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 机器人 $v_*$ | `optimal_v(P, 0.9)` | 16.949、15.254 |
| 往前看一步挑最大 | `greedy_actions(P, v, 0.9, s)` | 高电 ['搜索']、低电 ['充电'] |
| 网格 $v_*$ | `optimal_v(gridworld_5x5(), 0.9)` | 图 3.5 中（A 24.4） |
| 右下角并列最优 | `greedy_actions(G, vg, 0.9, 24)` | ['↑', '←'] |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `MOVE_NAMES`**（来自 `rl_envs.py`）

In [ ]:
MOVE_NAMES = ["↑", "↓", "→", "←"]

**常量 `ROBOT_ACTIONS`**（来自 `rl_envs.py`）

In [ ]:
ROBOT_ACTIONS = ["搜索", "等待", "充电"]

**常量 `ROBOT_STATES`**（来自 `rl_envs.py`）

In [ ]:
ROBOT_STATES = ["高电", "低电"]

**函数 `q_from_v`**（来自 `ch08_bellman.py`）：q_π(s, a) = Σ p(s', r|s, a) [r + γ v_π(s')]。

In [ ]:
def q_from_v(P, v, gamma, s, a):
    """q_π(s, a) = Σ p(s', r|s, a) [r + γ v_π(s')]。"""
    return sum(p * (r + (0 if done else gamma * v[ns])) for p, ns, r, done in P[s][a])

**函数 `greedy_actions`**（来自 `ch09_optimal.py`）：下面要用到的小工具。

In [ ]:
def greedy_actions(P, v, gamma, s, eps=1e-6):
    qs = {a: q_from_v(P, v, gamma, s, a) for a in P[s]}
    best = max(qs.values())
    return [a for a, q in qs.items() if q > best - eps]

**常量 `MOVES`**（来自 `rl_envs.py`）

In [ ]:
MOVES = [(-1, 0), (1, 0), (0, 1), (0, -1)]

**函数 `gridworld_5x5`**（来自 `rl_envs.py`）：S&B Example 3.5（图 3.2）：5×5，A=(0,1) 任何动作 +10 跳到 A'=(4,1)，B=(0,3) +5 跳到 B'=(2,3)，

In [ ]:
def gridworld_5x5():
    """S&B Example 3.5（图 3.2）：5×5，A=(0,1) 任何动作 +10 跳到 A'=(4,1)，B=(0,3) +5 跳到 B'=(2,3)，
    撞墙原地不动 −1，其余 0。连续任务，不结束。状态编号 = 行 × 5 + 列。"""
    n = 5
    P = {}
    for r in range(n):
        for c in range(n):
            s = r * n + c
            P[s] = {}
            for a, (dr, dc) in enumerate(MOVES):
                if (r, c) == (0, 1):
                    P[s][a] = [(1.0, 4 * n + 1, 10.0, False)]
                elif (r, c) == (0, 3):
                    P[s][a] = [(1.0, 2 * n + 3, 5.0, False)]
                else:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < n and 0 <= nc < n:
                        P[s][a] = [(1.0, nr * n + nc, 0.0, False)]
                    else:
                        P[s][a] = [(1.0, s, -1.0, False)]
    return P

**函数 `optimal_v`**（来自 `ch09_optimal.py`）：反复套最优方程直到不再变（这就是第 12 章的价值迭代）。

In [ ]:
def optimal_v(P, gamma, tol=1e-10):
    """反复套最优方程直到不再变（这就是第 12 章的价值迭代）。"""
    v = np.zeros(len(P))
    while True:
        new = np.array([max(q_from_v(P, v, gamma, s, a) for a in P[s]) for s in P])
        if np.max(np.abs(new - v)) < tol:
            return new
        v = new

**常量 `ALPHA`, `BETA`**（来自 `rl_envs.py`）

In [ ]:
ALPHA, BETA = 0.8, 0.7

**常量 `R_SEARCH`, `R_WAIT`, `R_RESCUE`**（来自 `rl_envs.py`）

In [ ]:
R_SEARCH, R_WAIT, R_RESCUE = 2.0, 1.0, -3.0

**函数 `robot_mdp`**（来自 `rl_envs.py`）：返回 P[s][a]；状态 0 = 高电、1 = 低电；动作 0 = 搜索、1 = 等待、2 = 充电（只有低电能充电）。

In [ ]:
def robot_mdp():
    """返回 P[s][a]；状态 0 = 高电、1 = 低电；动作 0 = 搜索、1 = 等待、2 = 充电（只有低电能充电）。"""
    hi, lo = 0, 1
    P = {
        hi: {
            0: [(ALPHA, hi, R_SEARCH, False), (1 - ALPHA, lo, R_SEARCH, False)],
            1: [(1.0, hi, R_WAIT, False)],
        },
        lo: {
            0: [(BETA, lo, R_SEARCH, False), (1 - BETA, hi, R_RESCUE, False)],
            1: [(1.0, lo, R_WAIT, False)],
            2: [(1.0, hi, 0.0, False)],
        },
    }
    return P

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
P = robot_mdp()

In [ ]:
v = optimal_v(P, 0.9)

In [ ]:
for s in P:
        acts = [ROBOT_ACTIONS[a] for a in greedy_actions(P, v, 0.9, s)]
        print(f"v*({ROBOT_STATES[s]}) = {v[s]:.3f}，最优动作 {acts}")

In [ ]:
G = gridworld_5x5()

In [ ]:
vg = optimal_v(G, 0.9)

In [ ]:
print("5×5 网格 v*（图 3.5 中）：")

In [ ]:
print(np.round(vg.reshape(5, 5), 1))

In [ ]:
print("右下角 (4,4) 的最优动作：", [MOVE_NAMES[a] for a in greedy_actions(G, vg, 0.9, 24)])

## 3. 代码里才看得到的

`optimal_v` 里那个 `while True` 循环，就是"从 0 开始反复套最优方程直到不再变"，这正是第 12 章的价值迭代；挑并列最优动作时用 `q > best − eps` 而不是 `==`，因为 16.000000001 和 16.0 在电脑里不相等。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. `optimal_v(P, 0.5)` → [3.667, 2.0]，这时 `greedy_actions(P, v, 0.5, 1)` 给出的是 ['等待']，不再是 ['充电']：γ 小了，智能体只顾眼前，充电的 0 分不如等待的 +1。最优动作跟着 γ 变。
2. 把 γ 调回 0.9 再看一遍低电的最优动作，回到 ['充电']。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 每个状态都不输，才算更好；最好的叫 π*；
> 期望方程按 π 平均，最优方程挑最大，环境那层照旧平均；
> 有了 v* 看一步，有了 q* 直接挑；
> 网格最优绕回 A，二十四点四。